# 31 Classification V2 — 지역 흐름: 상승 확률 대신 4주 흐름 순위 · 기회 신호

## 목적

설계(6-2)는 "구 × 월, 향후 3개월 +3% 상승 확률"을 분류 모델로 내는 것이었다. 조건은 하나였다:
**단순 기준선(최근 추세 유지)을 넘고 보정(calibration)이 확인될 때만 확률 숫자를 웹에 보여준다.**

## 결론 요약

- **정답을 바꿨다.** V1 정답(구 중위 평당가 3개월 변화)은 그 달 거래된 단지 구성에 흔들려 같은 단지 반복거래 지수와 상관이 0.14뿐이었다.
  부동산원 공식 **주간 아파트 매매가격지수(구별)의 순수 미래 3개월 변화**로 바꿨다 (06, 예측 시점 전에 공표된 주부터 3개월 뒤까지).
- **어떤 모델도 "최근 4주 흐름" 기준선을 Test에서 넘지 못했다.** 절대 정답(+1% · +3%)도, 상대 정답(같은 달 상위 8개 구)도 같다.
  시장 · 공급 · 인구 · 정책 · 회귀 OOF Gap · 구 정체성 · 주간 전세 · 수급 · 20년치 분기 지수를 더할수록 Test가 나빠졌다 (2026 국면 전환).

| 상대 정답: 같은 달 25개 구 중 향후 3개월 상위 8개 (무작위 적중 33%) | DEV 2023-11~2025-12 월내 AUC / 상위 8 적중 | Test 2026-01~06 |
|---|---:|---:|
| **기준선: 공식 지수 최근 4주 변화 순위** | 0.903 / 75.5% | **0.749 / 52.1%** |
| 로지스틱 · 공식 지수 3개 | 0.909 / 77.4% | 0.699 / 47.9% |
| LightGBM · 공식 지수 + 확정월 + 수급 + 공급 + 인구 + 회귀 OOF Gap | 0.917 / 77.4% | 0.512 / 25.0% |

- **결정: 확률 숫자는 보여주지 않는다.** 대신 웹 지역 화면에는
  1. **최근 4주 흐름 순위** (주간, 공식 지수 그대로 — 과거 적중률을 함께 표시)
  2. **기회 신호** = 11 적정가보다 5% 넘게 싸게 거래된 집 (+ 그 구의 4주 흐름) — 다음 거래에서 시장보다 평균 5.8~6.0%p 더 올랐다
  3. **정책 타임라인** (03 이벤트 · 이벤트 스터디)
  4. **권역 매매 · 전세 수급** (주간)
- 산출물: `models/classification/v2/` — 주간 흐름 표, 기회 신호 근거, 표시 규칙, 평가 지표

## 원칙

- 행 = 구 × 예측시점 p (매달 1일). Feature는 p 전에 공표된 값만, 정답이 확정된 행만 학습에 쓴다 (매달 다시 학습하는 walk-forward).
- DEV 2023-11~2025-12로 고르고 Test 2026-01~06은 확인만 한다.
- 실험(수십 개 조합 × 매달 재학습)은 `v2_experiments/` 스크립트로 돌렸고, 이 노트북은 결과를 불러와 해석한다.
  표시용 산출물(주간 흐름 순위 · 기준선 적중률 · 기회 신호 근거)은 여기서 직접 계산한다.

In [1]:
from pathlib import Path
import json

import numpy as np
import pandas as pd
from sklearn.metrics import roc_auc_score

pd.set_option("display.width", 200)
pd.set_option("display.max_columns", 40)


def find_project_root(start=Path.cwd()):
    for path in [start, *start.parents]:
        if (path / "data").exists() and (path / ".env").exists():
            return path
    raise FileNotFoundError("프로젝트 루트를 찾을 수 없습니다.")


PROJECT_DIR = find_project_root()
PROCESSED_DIR = PROJECT_DIR / "data" / "v2_5year_seoul" / "processed"
WEEKLY_DIR = PROJECT_DIR / "data" / "external" / "rone_weekly"
EXP_DIR = PROJECT_DIR / "notebook" / "v2_5year_notebook" / "model" / "classification" / "v2_experiments"
MODEL_DIR = PROJECT_DIR / "models" / "classification" / "v2"
MODEL_DIR.mkdir(parents=True, exist_ok=True)

region = pd.read_parquet(PROCESSED_DIR / "v2_region_month.parquet")
region["p"] = region["예측시점"].dt.to_period("M")
DEV = pd.period_range("2023-11", "2025-12", freq="M")
TEST = pd.period_range("2026-01", "2026-06", freq="M")
print(region.shape, region["p"].min(), "~", region["p"].max())
region.groupby(region["p"].dt.year)[["상승세", "강한상승"]].mean().round(3).rename(columns={"상승세": "+1% 비율", "강한상승": "+3% 비율"}).T

(1500, 149) 2021-07 ~ 2026-06


p,2021,2022,2023,2024,2025,2026
+1% 비율,0.600,0.0,0.193,0.423,0.667,0.907
+3% 비율,0.064,0.0,0.007,0.100,0.313,0.380


## 1. 정답 정의

V1 정답은 "구 중위 평당가의 향후 3개월 변화 > 3%"였다. 중위 평당가는 그 달 **어떤 단지가 거래됐는지**에 따라 움직인다.

- 같은 단지 · 면적 반복거래로 만든 구 지수와 비교하면 변화율 상관 0.14, 정답 일치율 72%. 같은 구에서 예측 시점에 아는 최신 정답과의 자기상관은 -0.19 → 사실상 잡음 (`r31_target_diag.py`)
- 부동산원 공식 주간 지수: 반복거래 지수와 상관 0.57, 같은 구 자기상관 0.96 (`r31_official_diag.py`)
- 그래서 06에서 정답을 **공식 지수의 순수 미래 3개월 변화**로 바꿨다: 예측 시점 p 전에 공표된 최신 주 → 3개월 뒤 같은 시점 (`공식_향후3개월_변화율`).
  `상승세` = +1% 초과, `강한상승` = +3% 초과 (+3%는 DEV 상승 비율 19%로 드물다)

In [2]:
for name in ["r31_target_diag", "r31_official_diag", "r31_forward_diag"]:
    print(f"── {name}")
    print(open(EXP_DIR / f"{name}_output.txt", encoding="utf-8").read().split("           강남구")[0])

── r31_target_diag
중위 기준 vs 반복거래 기준 변화율 상관 0.142 | 정답 일치율 0.717 | 상승 비율 중위 0.383 반복거래 0.2
향후3개월_변화율 같은 구 자기상관 lag1 0.611 lag4(예측 시점에 알 수 있는 최신) -0.186
rs_변화 같은 구 자기상관 lag1 0.889 lag4(예측 시점에 알 수 있는 최신) 0.174
DEV 기준선 AUC — 반복거래 모멘텀(≤p-2)으로 반복거래 정답: 0.69 | 같은 점수로 중위 정답: 0.662

── r31_official_diag
행 1375 | p 2021-10 ~ 2026-04
3개월 변화율 분포 (공식): {'count': 1375.0, 'mean': 0.0033, 'std': 0.0234, 'min': -0.0932, '25%': -0.0025, '50%': 0.0044, '75%': 0.0157, '90%': 0.0294, 'max': 0.0649}
임계 0.030: 상승 비율 공식 0.097 | 반복거래 0.173
임계 0.020: 상승 비율 공식 0.199 | 반복거래 0.317
임계 0.015: 상승 비율 공식 0.266 | 반복거래 0.394
임계 0.010: 상승 비율 공식 0.362 | 반복거래 0.474
임계 0.005: 상승 비율 공식 0.484 | 반복거래 0.559
상관 — 공식 vs 반복거래 0.571 | 공식 vs 중위 -0.117
공식 자기상관 lag1 0.955 lag4 0.505
DEV 임계 0.03: 상승 비율 0.165 | 공식 모멘텀(≤p-1) AUC 0.926 | 반복거래 모멘텀(≤p-2) AUC 0.813
DEV 임계 0.01: 상승 비율 0.475 | 공식 모멘텀(≤p-1) AUC 0.855 | 반복거래 모멘텀(≤p-2) AUC 0.787

── r31_forward_diag
현재 정답 변화 중 예측 시점에 이미 일어난 부분 비중 (중앙): 0.23
상관: 현재 정답 vs 순수 미래 1.0
+1% 현재 정답: 상승 비율 

## 2. 1단계 — V1 정답 재현 · Feature 묶음 (`r31_stage1.py`)

V1 정답으로 V1 결과(18개월 rolling AUC 0.613)를 재현한 뒤, 같은 정답에서 V2 Feature 묶음 · 모델 · 학습 창을 비교했다 (DEV, ROC-AUC).

In [3]:
s1 = pd.read_csv(EXP_DIR / "r31_stage1.csv")
print(s1.loc[0, ["구성", "ROC_AUC", "비고"]].to_dict())
base1 = s1[s1["구성"].str.startswith("기준선")][["구성", "ROC_AUC", "월내_AUC"]]
s1m = s1[s1["구성"].str.contains(r"\|")].copy()
s1m[["묶음", "모델", "창"]] = s1m["구성"].str.split(r" \| ", expand=True)
display(base1.round(3))
s1m.pivot_table(index="묶음", columns=["모델", "창"], values="ROC_AUC", sort=False).round(3)

{'구성': 'V1 재현 (V1 fold 4개, 18개월)', 'ROC_AUC': 0.6128639695477802, '비고': '[0.742, 0.72, 0.567, 0.422]'}


,구성,ROC_AUC,월내_AUC
1,기준선: 학습 기간 상승 비율,0.363,0.500
2,기준선: 최근 확정 3개월 변화율,0.399,0.439
3,기준선: 서울 상승 구 비율,0.478,0.500
4,기준선: 같은 구 최근 확정 정답 (p-4),0.426,0.480


모델,XGB(V1 설정),LightGBM,로지스틱,XGB(V1 설정),LightGBM,로지스틱,LightGBM
창,창 전체,창 전체,창 전체,창 18,창 18,창 18,창 36
묶음,,,,,,,
V1 Feature (당월 거래),0.506,0.541,0.585,0.515,0.517,0.508,0.534
V1 정의 + 확정월 (시장·전월세·금리),0.493,0.517,0.496,0.501,0.499,0.467,0.517
+ 시장지표(01),0.462,0.471,0.533,0.520,0.493,0.503,0.480
+ 정책(03),0.469,0.500,0.551,0.540,0.503,0.573,0.509
+ 공급(04),0.466,0.502,0.549,0.518,0.501,0.600,0.484
+ 인구(05) = 전체,0.480,0.508,0.542,0.494,0.470,0.603,0.485
+ 회귀 OOF Gap,0.464,0.491,0.571,0.512,0.507,0.607,0.492


- V1 정답은 기준선(0.36~0.48)도 모델(대부분 0.46~0.61)도 동전 던지기 수준이다. V1의 0.61도 fold별 0.42~0.74로 흔들린 값이다.
- Feature를 더해도, 모델 · 학습 창을 바꿔도 일관된 개선이 없다 → **정답 자체가 예측할 수 없는 잡음**이라는 1장 진단과 맞는다.

## 3. 2 · 3단계 — 공식 정답 (절대: +1% · +3%)

`r31_stage2.py`: Feature 묶음 사다리 × 모델 × 학습 창 (DEV). `r31_stage3.py`: 최근 흐름 + 작은 Feature 조합 (DEV · Test).

In [4]:
for target in ["상승세", "강한상승"]:
    s2 = pd.read_csv(EXP_DIR / f"r31_stage2_{target}.csv")
    s2m = s2[s2["구성"].str.contains(r"\|")].copy()
    s2m[["묶음", "모델", "창"]] = s2m["구성"].str.split(r" \| ", expand=True)
    print(f"── 정답 {target} (DEV 상승 비율 {s2['상승비율'].iloc[0]:.3f}) — 기준선")
    display(s2[s2["구성"].str.startswith("기준선")][["구성", "ROC_AUC", "월내_AUC", "Brier"]].round(3))
    display(s2m.pivot_table(index="묶음", columns=["모델", "창"], values="ROC_AUC", sort=False).round(3))

── 정답 상승세 (DEV 상승 비율 0.503) — 기준선


,구성,ROC_AUC,월내_AUC,Brier
0,기준선: 학습 기간 상승 비율,0.598,0.500,0.316
1,기준선: 공식지수_4주변화 순위,0.837,0.916,0.165
2,기준선: 공식지수_13주변화 순위,0.723,0.937,0.222
3,기준선: 공식지수만 로지스틱,0.729,0.864,0.210


모델,LightGBM,로지스틱,LightGBM,로지스틱,LightGBM,로지스틱
창,창 전체,창 전체,창 36,창 36,창 24,창 24
묶음,,,,,,
V1 Feature (당월 거래),0.884,0.783,0.887,0.898,0.884,0.916
V1 정의 + 확정월,0.852,0.851,0.819,0.847,0.822,0.840
공식지수 + 확정월,0.834,0.849,0.801,0.844,0.824,0.834
+ 시장지표(01),0.836,0.841,0.811,0.840,0.813,0.812
+ 정책(03),0.842,0.828,0.817,0.847,0.818,0.837
+ 공급(04),0.853,0.842,0.824,0.853,0.812,0.846
+ 인구(05),0.853,0.840,0.818,0.856,0.818,0.844
+ 회귀 OOF Gap (전체),0.854,0.840,0.812,0.856,0.812,0.843


── 정답 강한상승 (DEV 상승 비율 0.191) — 기준선


,구성,ROC_AUC,월내_AUC,Brier
0,기준선: 학습 기간 상승 비율,0.637,0.500,0.175
1,기준선: 공식지수_4주변화 순위,0.863,0.935,0.222
2,기준선: 공식지수_13주변화 순위,0.824,0.919,0.234
3,기준선: 공식지수만 로지스틱,0.721,0.931,0.135


모델,LightGBM,로지스틱,LightGBM,로지스틱,LightGBM,로지스틱
창,창 전체,창 전체,창 36,창 36,창 24,창 24
묶음,,,,,,
V1 Feature (당월 거래),0.864,0.730,0.871,0.767,0.858,0.829
V1 정의 + 확정월,0.842,0.783,0.821,0.781,0.797,0.736
공식지수 + 확정월,0.823,0.780,0.807,0.779,0.781,0.741
+ 시장지표(01),0.828,0.770,0.820,0.768,0.778,0.718
+ 정책(03),0.817,0.723,0.811,0.716,0.780,0.689
+ 공급(04),0.826,0.795,0.824,0.787,0.796,0.764
+ 인구(05),0.827,0.790,0.826,0.784,0.799,0.764
+ 회귀 OOF Gap (전체),0.824,0.789,0.820,0.784,0.795,0.763


- DEV에서는 공식 지수 **최근 4주 변화 순위만으로** ROC-AUC 0.837(+1%) · 0.863(+3%)이다. 확정월 시장 · 수급 · 정책 · 공급 · 인구 · OOF Gap을 더한 모델은 0.69~0.86으로 비슷하거나 낮다.
- `V1 Feature (당월 거래)`가 가장 높게(LightGBM 0.86~0.89, 로지스틱 최고 0.92) 나오지만, **예측 시점에 아직 신고가 다 안 된 당월 거래**를 쓰는 Feature라 서비스에서 만들 수 없다 (5장 시점 규칙).
  같은 정의를 확정월로 바꾸면(`V1 정의 + 확정월`) 0.74~0.85로 내려간다 → V1 분류 성능의 일부는 신고 시차 위반에서 나왔다.
- 회귀 OOF Gap은 어느 조합에서도 일관된 개선이 없다 (11 보완 2 OOF로 다시 돌려도 AUC 차이 ≤ 0.03).

In [5]:
rows = []
for target in ["상승세", "강한상승"]:
    s3 = pd.read_csv(EXP_DIR / f"r31_stage3_{target}.csv")
    rows.append(s3.pivot_table(index=["Feature", "모델"], columns="기간", values=["ROC_AUC", "월내_AUC"], sort=False)
                .reorder_levels([1, 0], axis=1).sort_index(axis=1, ascending=[True, False]).assign(정답=target))
    print(target, "Test 상승 비율", round(s3.loc[s3["기간"].eq("Test"), "상승비율"].iloc[0], 3))
pd.concat(rows).round(3)

상승세 Test 상승 비율 0.907
강한상승 Test 상승 비율 0.38


기간                                      DEV           Test            정답
                                     월내_AUC ROC_AUC 월내_AUC ROC_AUC      
Feature                모델                                               
4주 변화율만                로지스틱 창 전체      0.916   0.782  0.744   0.625   상승세
                       로지스틱 창 36      0.916   0.779  0.744   0.638   상승세
4주 + 13주               로지스틱 창 전체      0.842   0.805  0.779   0.654   상승세
                       로지스틱 창 36      0.913   0.775  0.729   0.629   상승세
4주 + 서울 흐름             로지스틱 창 전체      0.916   0.769  0.744   0.624   상승세
                       로지스틱 창 36      0.916   0.722  0.744   0.637   상승세
                       LightGBM 창 전체  0.909   0.750  0.716   0.550   상승세
4주 + 확정월 구 시장          로지스틱 창 전체      0.875   0.813  0.466   0.600   상승세
                       로지스틱 창 36      0.916   0.830  0.367   0.541   상승세
                       LightGBM 창 전체  0.911   0.847  0.527   0.573   상승세
4주 + 확정월 구 시장 + 금리     로지스틱 창 전체      0.947   0.832  0.452   0.662   상승세
                       로지스틱 창 36      0.936   0.845  0.311   0.511   상승세
                       LightGBM 창 전체  0.927   0.843  0.481   0.526   상승세
4주 + 수급·심리(01)         로지스틱 창 전체      0.834   0.778  0.710   0.711   상승세
                       로지스틱 창 36      0.844   0.792  0.677   0.642   상승세
                       LightGBM 창 전체  0.849   0.840  0.701   0.555   상승세
공식지수 + 확정월 (2단계 최고 묶음) 로지스틱 창 전체      0.943   0.845  0.470   0.724   상승세
                       로지스틱 창 36      0.943   0.844  0.327   0.582   상승세
                       LightGBM 창 전체  0.906   0.834  0.438   0.486   상승세
4주 변화율만                로지스틱 창 전체      0.935   0.792  0.761   0.616  강한상승
                       로지스틱 창 36      0.935   0.761  0.761   0.637  강한상승
4주 + 13주               로지스틱 창 전체      0.905   0.779  0.755   0.594  강한상승
                       로지스틱 창 36      0.933   0.755  0.754   0.602  강한상승
4주 + 서울 흐름             로지스틱 창 전체      0.935   0.762  0.761   0.632  강한상승
                       로지스틱 창 36      0.935   0.710  0.761   0.616  강한상승
                       LightGBM 창 전체  0.875   0.713  0.767   0.626  강한상승
4주 + 확정월 구 시장          로지스틱 창 전체      0.885   0.753  0.549   0.610  강한상승
                       로지스틱 창 36      0.899   0.805  0.563   0.607  강한상승
                       LightGBM 창 전체  0.888   0.843  0.433   0.463  강한상승
4주 + 확정월 구 시장 + 금리     로지스틱 창 전체      0.905   0.733  0.609   0.681  강한상승
                       로지스틱 창 36      0.905   0.799  0.522   0.508  강한상승
                       LightGBM 창 전체  0.892   0.849  0.477   0.499  강한상승
4주 + 수급·심리(01)         로지스틱 창 전체      0.868   0.677  0.693   0.725  강한상승
                       로지스틱 창 36      0.868   0.749  0.668   0.682  강한상승
                       LightGBM 창 전체  0.856   0.803  0.668   0.688  강한상승
공식지수 + 확정월 (2단계 최고 묶음) 로지스틱 창 전체      0.906   0.742  0.497   0.650  강한상승
                       로지스틱 창 36      0.880   0.779  0.440   0.490  강한상승
                       LightGBM 창 전체  0.890   0.823  0.424   0.515  강한상승

- **Test(2026 상반기)는 서울 거의 전체가 올랐다** (+1% 기준 90.7%). 절대 정답으로는 "어느 구가 오르나"를 가릴 수 없고, AUC가 몇 개월 행에 좌우된다.
- 그래도 같은 경향이다: 최근 흐름(4주 · 13주)만 쓴 로지스틱이 Test 월내 AUC 0.74~0.78, 확정월 시장 · 금리를 더하면 0.31~0.61로 무너진다.
- → 시장 전체 방향과 상관없이 매달 평가할 수 있는 **상대 정답**으로 다시 봤다 (4장).

## 4. 4단계 — 상대 정답: 같은 달 상위 8개 구 (`r31_stage4.py`)

같은 달 25개 구 중 향후 3개월 상승률 상위 1/3(8개)을 맞히는가. 매달 정답 비율이 같아 2026처럼 모두 오른 기간도 평가할 수 있다.
Feature는 같은 달 서울 평균과의 차이로 바꿨다 (서울 공통 값은 0이 돼 빠진다).

지표: 월내 AUC(같은 달 안에서 구 순위), 상위 8 적중률(모델 상위 8개 중 실제 상위 8개 비율, 무작위 33%), 상위 8 − 하위 8 실제 상승률 차.

In [6]:
s4 = pd.read_csv(EXP_DIR / "r31_stage4_상대상위.csv")
s4.pivot_table(index="방법", columns="기간", values=["월내_AUC", "상위8_적중률", "상위8-하위8_상승률차(%p)"], sort=False) \
  .reorder_levels([1, 0], axis=1).sort_index(axis=1, ascending=[True, False]).round(3)

기간                                               DEV                            Test                         
                                              월내_AUC 상위8_적중률 상위8-하위8_상승률차(%p) 월내_AUC 상위8_적중률 상위8-하위8_상승률차(%p)
방법                                                                                                           
기준선: 공식지수_4주변화                                 0.903   0.755            1.987  0.749   0.521            1.112
기준선: 공식지수_13주변화                                0.899   0.769            2.001  0.635   0.354            0.722
기준선: 공식지수_26주변화                                0.886   0.750            1.981  0.504   0.250            0.184
로지스틱 창 전체 · 공식지수                               0.909   0.774            2.040  0.699   0.479            0.985
로지스틱 창 36 · 공식지수                               0.909   0.779            2.045  0.695   0.479            0.985
LightGBM 창 전체 · 공식지수                           0.890   0.740            1.693  0.673   0.458            0.705
로지스틱 창 전체 · 공식지수 + 확정월 시장·전월세                  0.910   0.769            2.033  0.566   0.354           -0.056
로지스틱 창 36 · 공식지수 + 확정월 시장·전월세                  0.910   0.769            2.034  0.542   0.271           -0.147
LightGBM 창 전체 · 공식지수 + 확정월 시장·전월세              0.911   0.764            1.947  0.561   0.333           -0.038
로지스틱 창 전체 · 공식지수 + 확정월 + 수급(01) + 공급 + 인구      0.882   0.760            1.877  0.558   0.312           -0.185
로지스틱 창 36 · 공식지수 + 확정월 + 수급(01) + 공급 + 인구      0.887   0.760            1.910  0.588   0.333           -0.019
LightGBM 창 전체 · 공식지수 + 확정월 + 수급(01) + 공급 + 인구  0.916   0.774            1.957  0.529   0.271           -0.179
로지스틱 창 전체 · 위 + 회귀 OOF Gap                     0.882   0.764            1.872  0.562   0.292           -0.229
로지스틱 창 36 · 위 + 회귀 OOF Gap                     0.889   0.769            1.937  0.585   0.333           -0.047
LightGBM 창 전체 · 위 + 회귀 OOF Gap                 0.917   0.774            1.954  0.512   0.250           -0.152

- DEV에서는 모델(0.88~0.92)이 기준선(0.89~0.90)과 거의 같다. Test에서는 **기준선 4주 흐름(0.749 · 적중 52%)이 가장 좋고**, Feature를 더할수록 나빠진다 (확정월 이상 0.51~0.59, 적중 25~35%).
- 13주 · 26주 흐름은 Test에서 0.63 · 0.50 — 2026 상반기 상승 구가 바뀌어 긴 흐름일수록 틀렸다.

## 5. 추가 점검 — 더 많은 정보가 도움이 되나

| 점검 | 스크립트 | 질문 |
|---|---|---|
| 구 고정 효과 | `r31_gu_effect.py` | 구 이름(원-핫)을 더하면? 상권 · 학군처럼 천천히 변하는 구 특성이 줄 수 있는 효과의 상한 |
| 주간 전세 · 수급 | `r31_weekly_extra.py` | 빨리 변하는 주간 지표(구별 전세 지수, 권역 매매 · 전세 수급)를 더하면? |
| 20년 분기 지수 | `r31_long_quarterly.py` | 2006년부터의 구별 분기 실거래가격지수(KOSIS)로 더 긴 이력을 학습하면? "덜 오른 구가 따라 오른다"(갭 메우기)는? |

In [7]:
ge = pd.read_csv(EXP_DIR / "r31_gu_effect.csv")
display(ge[ge["C"].eq(0.1)].pivot_table(index="Feature", columns=["정답", "기간"], values="월내_AUC", sort=False).round(3))
we = pd.read_csv(EXP_DIR / "r31_weekly_extra.csv")
display(we.pivot_table(index="방법", columns=["정답", "기간"], values="월내_AUC", sort=False).round(3))
lq = pd.read_csv(EXP_DIR / "r31_long_quarterly.csv").rename(columns={"Unnamed: 0": "방법"}).set_index("방법")
display(lq.round(3))
print(open(EXP_DIR / "r31_long_quarterly_output.txt", encoding="utf-8").read().strip().splitlines()[-1])

정답          상승세(+1%)        상대 상위 1/3       
기간               DEV   Test       DEV   Test
Feature                                     
공식지수           0.864  0.749     0.911  0.668
공식지수 + 구 이름    0.919  0.570     0.889  0.332
구 이름만          0.928  0.301     0.878  0.304

정답                                 상승세(+1%)        상대 상위 1/3       
기간                                      DEV   Test       DEV   Test
방법                                                                 
기준선: 4주 흐름                            0.916  0.744     0.903  0.749
로지스틱 · 공식지수                           0.864  0.749     0.911  0.668
LightGBM · 공식지수                       0.827  0.711     0.858  0.668
로지스틱 · 공식지수 + 주간 전세                   0.864  0.750     0.907  0.668
LightGBM · 공식지수 + 주간 전세               0.876  0.730     0.838  0.665
로지스틱 · 공식지수 + 주간 수급                   0.883  0.748     0.911  0.675
LightGBM · 공식지수 + 주간 수급               0.847  0.783     0.840  0.663
로지스틱 · 공식지수 + 주간 전세 · 수급              0.911  0.743     0.909  0.676
LightGBM · 공식지수 + 주간 전세 · 수급          0.869  0.740     0.838  0.662
로지스틱 · 공식지수 + 확정월 + 주간 전세 · 수급        0.942  0.407     0.902  0.433
LightGBM · 공식지수 + 확정월 + 주간 전세 · 수급    0.899  0.551     0.889  0.371

,전체 2012~2026,2012~2016,2017~2021,2022~2024,2025~2026,상위8 적중률(전체)
방법,,,,,,
기준선: 1분기변화,0.544,0.519,0.543,0.562,0.625,0.364
기준선: 2분기변화,0.539,0.505,0.513,0.613,0.614,0.375
기준선: 4분기변화,0.553,0.553,0.490,0.640,0.608,0.375
로지스틱 · 최근 흐름만 (1·2·4분기 순위),0.529,0.514,0.508,0.563,0.614,0.346
LightGBM · 최근 흐름만 (1·2·4분기 순위),0.532,0.533,0.512,0.598,0.434,0.346
로지스틱 · 최근 흐름 + 갭 메우기,0.536,0.521,0.526,0.559,0.594,0.355
LightGBM · 최근 흐름 + 갭 메우기,0.507,0.549,0.450,0.555,0.447,0.317
로지스틱 · 최근 흐름 + 갭 메우기 + 서울 국면,0.537,0.521,0.526,0.560,0.594,0.355
LightGBM · 최근 흐름 + 갭 메우기 + 서울 국면,0.528,0.564,0.486,0.562,0.460,0.350


갭 메우기 단독(지난 3년 덜 오른 구 = 높은 점수) 분기내 AUC: {'전체 2012~2026': np.float64(0.483), '2012~2016': np.float64(0.51), '2017~2021': np.float64(0.497), '2022~2024': np.float64(0.449), '2025~2026': np.float64(0.388)}


- **구 이름:** DEV에서는 구 이름만으로도 월내 AUC 0.88~0.93 ("늘 오르는 구"), Test에서는 0.30 — 2026에는 그동안 덜 오르던 구가 올랐다. 구 정체성은 오히려 반대로 작동했다.
  → 상권 · 학군 같은 느리게 변하는 구 특성 Feature도 기대하기 어렵다.
- **주간 전세 · 수급:** Test 상대 정답에서 0.66~0.68로 기준선(0.749)에 못 미친다. 확정월까지 더하면 0.37~0.43.
- **20년 분기 지수:** 2012~2026 전체에서 모델 0.51~0.54 < 최근 흐름 기준선 0.54~0.55. 갭 메우기 단독은 0.48, 2025~2026은 0.39로 오히려 반대다.
- 결론: 데이터 양 · 종류가 아니라 **3개월 뒤 구 순위 자체가 최근 흐름 이상으로는 예측되지 않는다.** 기준선을 넘는 모델이 없으니 6-2의 노출 조건(기준선 돌파 + 보정)을 충족하지 못한다.

## 6. 서비스 지표 ① — 최근 4주 흐름 순위

확률 대신 **공식 지수의 최근 4주 변화 순위**를 그대로 보여준다. 모델이 아니라 공표값이라 설명이 쉽고, 위 실험에서 가장 강한 신호였다.

- 매주 목요일(기준일 월요일 + 3일) 공표 → 그 주 순위 갱신. 매매 4주 · 13주 변화, 전세 4주 변화, 권역 수급을 함께 둔다.
- 4주 변화는 `log(지수_t / 지수_t-4)` (t-4 = 공표 4회 전, 설 · 추석 주는 공표 없음), 화면에는 `(exp − 1) × 100` %로 보여준다.
- 아래에서 06의 `공식지수_4주변화`(평가에 쓴 값)와 순위가 같은지 확인하고, 과거 적중률을 다시 계산한다.

In [8]:
def weekly(name, value="지수"):
    d = pd.read_csv(WEEKLY_DIR / name, dtype={"주": str})
    d["주"] = pd.to_datetime(d["주"])
    return d

sale = np.log(weekly("apt_sale_index_weekly_gu.csv").pivot(index="주", columns="자치구", values="지수").sort_index())
jeonse = np.log(weekly("apt_jeonse_index_weekly_gu.csv").pivot(index="주", columns="자치구", values="지수").sort_index())
sd = weekly("apt_supply_demand_weekly_seoul.csv")
# 설 · 추석 주는 공표가 없다 (2025-01-27 · 2025-10-06) → 06과 같이 "4주" = 공표 4회 전 (shift)
assert sale.shape[1] == 25 and (sale.index.dayofweek == 0).all() and sale.index.equals(jeonse.index)
print("공표 간격:", sale.index.to_series().diff().value_counts().to_dict())

trend = pd.concat({"매매_4주": sale - sale.shift(4), "매매_13주": sale - sale.shift(13), "전세_4주": jeonse - jeonse.shift(4)}, axis=1)
trend = trend.stack(level=1, future_stack=True).reset_index().rename(columns={"level_1": "자치구"}).dropna(subset=["매매_4주"])
trend["공표일"] = trend["주"] + pd.Timedelta(days=3)
trend["4주순위"] = trend.groupby("주")["매매_4주"].rank(ascending=False, method="min").astype(int)
trend["최근많이오른구"] = trend["4주순위"] <= 8
for c in ["매매_4주", "매매_13주", "전세_4주"]:
    trend[f"{c}_%"] = (np.exp(trend[c]) - 1) * 100
gu_zone = region[["자치구", "권역"]].drop_duplicates()
assert gu_zone["자치구"].is_unique and len(gu_zone) == 25
trend = trend.merge(gu_zone, on="자치구", how="left").merge(
    sd[["권역", "주", "매매수급", "전세수급"]], on=["권역", "주"], how="left", validate="many_to_one")
assert trend["권역"].notna().all()

# 06 평가값과 같은 순위인지: 예측시점 p 1일 전에 공표된 최신 주의 순위
check = []
for p, g in region.groupby("예측시점"):
    wk = trend.loc[trend["공표일"] < p, "주"].max()
    if pd.isna(wk) or g["공식지수_4주변화"].isna().any():
        continue
    a = g.set_index("자치구")["공식지수_4주변화"].rank(ascending=False, method="min")
    b = trend[trend["주"].eq(wk)].set_index("자치구")["4주순위"]
    check.append((a.reindex(b.index) == b).all())
assert all(check), "06 공식지수_4주변화와 주간 순위가 다름"
print(f"06 평가값과 순위 일치: {len(check)}개 예측시점 | 주간 표 {trend['주'].min():%Y-%m-%d} ~ {trend['주'].max():%Y-%m-%d}, {len(trend):,}행")
trend[trend["주"].eq(trend["주"].max())].sort_values("4주순위")[["자치구", "권역", "4주순위", "매매_4주_%", "매매_13주_%", "전세_4주_%", "매매수급", "전세수급"]].round(2).head(10)

공표 간격: {Timedelta('7 days 00:00:00'): 272, Timedelta('14 days 00:00:00'): 2}


06 평가값과 순위 일치: 53개 예측시점 | 주간 표 2021-08-02 ~ 2026-10-19, 6,775행


C:\Users\UK\AppData\Local\Temp\ipykernel_67816\1049194573.py:15: DeprecationWarning: The 'generic' unit for NumPy timedelta is deprecated, and will raise an error in the future. This includes implicit conversion of bare integers (e.g. `+ 1`).Please use a specific unit instead.
  trend["공표일"] = trend["주"] + pd.Timedelta(days=3)


,자치구,권역,4주순위,매매_4주_%,매매_13주_%,전세_4주_%,매매수급,전세수급
6763,서대문구,서북권,1,1.54,5.56,1.05,113.11,113.93
6766,성북구,동북권,2,1.43,5.67,1.42,118.49,123.18
6760,동대문구,동북권,3,1.32,4.47,0.83,118.49,123.18
6758,노원구,동북권,4,1.29,5.07,1.36,118.49,123.18
6759,도봉구,동북권,5,1.27,4.48,1.10,118.49,123.18
6752,강북구,동북권,6,1.25,4.89,0.82,118.49,123.18
6757,금천구,서남권,7,1.21,4.10,1.11,113.63,126.94
6774,중랑구,동북권,8,1.19,5.62,1.15,118.49,123.18
6754,관악구,서남권,9,1.15,4.27,0.70,113.63,126.94
6756,구로구,서남권,10,1.08,3.93,0.94,113.63,126.94


In [9]:
# 과거 적중률: 매달 1일 4주 흐름 상위 8개 구 중 향후 3개월 실제 상위 8개에 든 비율 (기준선 재계산 — 4장 값과 같아야 함)
r = region.dropna(subset=["공식지수_4주변화", "공식_향후3개월_변화율"]).copy()
r["실제상위8"] = r.groupby("p")["공식_향후3개월_변화율"].rank(ascending=False, method="first") <= 8
monthly = []
for p, g in r.groupby("p"):
    top = g.nlargest(8, "공식지수_4주변화")
    monthly.append({"p": p, "월내_AUC": roc_auc_score(g["실제상위8"], g["공식지수_4주변화"]), "상위8_적중률": top["실제상위8"].mean(),
                    "상위8_평균상승률_%": top["공식_향후3개월_변화율"].mean() * 100, "서울_평균상승률_%": g["공식_향후3개월_변화율"].mean() * 100})
monthly = pd.DataFrame(monthly)
monthly["구간"] = np.select([monthly["p"].isin(DEV), monthly["p"].isin(TEST)], ["DEV", "Test"], "그 이전")
hit = monthly.groupby("구간")[["월내_AUC", "상위8_적중률", "상위8_평균상승률_%", "서울_평균상승률_%"]].mean()
assert abs(hit.loc["DEV", "월내_AUC"] - s4.loc[s4["방법"].eq("기준선: 공식지수_4주변화") & s4["기간"].eq("DEV"), "월내_AUC"].iloc[0]) < 1e-9
assert abs(hit.loc["Test", "월내_AUC"] - s4.loc[s4["방법"].eq("기준선: 공식지수_4주변화") & s4["기간"].eq("Test"), "월내_AUC"].iloc[0]) < 1e-9
display(hit.round(3))
monthly.groupby(monthly["p"].dt.year)[["월내_AUC", "상위8_적중률", "상위8_평균상승률_%", "서울_평균상승률_%"]].mean().round(3)

,월내_AUC,상위8_적중률,상위8_평균상승률_%,서울_평균상승률_%
구간,,,,
DEV,0.903,0.755,2.538,1.487
Test,0.749,0.521,3.120,2.672
그 이전,0.867,0.696,-0.767,-1.428


,월내_AUC,상위8_적중률,상위8_평균상승률_%,서울_평균상승률_%
p,,,,
2022,0.863,0.693,-1.703,-2.391
2023,0.876,0.698,0.217,-0.343
2024,0.879,0.750,1.769,1.122
2025,0.927,0.771,3.732,2.136
2026,0.749,0.521,3.120,2.672


- 기준선 재계산이 4장 값과 같다 (assert). 2022~2025는 매년 4주 흐름 상위 8개 구의 69~77%가 다음 3개월에도 상위 8개에 들었고, 매년 서울 평균보다 0.6~1.6%p 더 올랐다.
- 다만 2026 상반기처럼 국면이 바뀌면 적중률이 52%까지 내려간다. 화면 문구는 "최근 4주 많이 오른 구"(사실)로 하고, 미래 확률처럼 쓰지 않는다. 과거 적중률은 연도별로 함께 보여준다.

## 7. 서비스 지표 ② — 기회 신호 (31 × 11, `r31_cheap_rising.py`)

"많이 오를 구에서 적정가보다 싸게 거래된 집"이 실제로 그 뒤 더 올랐나. 같은 단지 · 같은 면적의 다음 중개거래(90~400일 뒤)와 비교했다.

- 싸게/비싸게 = 첫 거래의 11 OOF 잔차 `r = log(실거래 / 적정가)` (매달 재학습 예측 → 그 시점에 알 수 있던 적정가, 11 보완 2 OOF로 다시 계산)
- 초과상승 = 두 거래 사이 가격 변화 − 같은 기간 그 구 공식 지수 변화 (시장 흐름을 뺀 값)
- 많이 오를 구 = 첫 거래 달 1일 기준 4주 흐름 상위 8개 구

In [10]:
print(open(EXP_DIR / "r31_cheap_rising_output.txt", encoding="utf-8").read().split("[구 흐름 × 가격]")[0].split("\n", 1)[-1].strip()[:200])
cheap = pd.read_csv(EXP_DIR / "r31_cheap_rising.csv")
cheap.pivot_table(index="가격", columns="구 흐름", values=["가격변화(%)", "시장변화(%)", "초과상승(%p)"], sort=False) \
     .reindex(["5% 넘게 쌈", "2~5% 쌈", "적정(±2%)", "2~5% 비쌈", "5% 넘게 비쌈"]).round(2)

wk['공표일'] = pd.to_datetime(wk['주']) + pd.Timedelta(days=3); wk['log지수'] = np.log(wk['지수'])
C:\Users\uk\Apartment price\notebook\v2_5year_notebook\model\classification\v2_experiments\r31_cheap_rising.p


가격변화(%)               시장변화(%)               초과상승(%p)              
구 흐름       나머지 구 많이 오를 구(상위 8)   나머지 구 많이 오를 구(상위 8)    나머지 구 많이 오를 구(상위 8)
가격                                                                         
5% 넘게 쌈     6.06          8.21    0.06          2.40     6.00          5.81
2~5% 쌈      4.43          6.33    1.31          3.27     3.13          3.07
적정(±2%)     3.70          5.57    1.60          3.45     2.10          2.12
2~5% 비쌈     2.62          5.07    1.76          3.61     0.87          1.45
5% 넘게 비쌈    0.91          3.40    1.92          3.84    -1.01         -0.44

- **적정가보다 5% 넘게 싸게 거래된 집은 다음 거래에서 시장보다 평균 5.8~6.0%p 더 올랐다** (적정 ±2%는 2.1%p, 5% 넘게 비싼 집은 -0.4 ~ -1.0%p). 2022~2026 매년 같은 방향이다 (5% 넘게 쌈 − 적정 = 4.2~8.7%p).
  → 11 적정가가 실제로 가격이 돌아갈 자리를 짚는다는 근거다.
- 초과상승 크기는 구 흐름과 상관없이 비슷하다 (상위 8 구 5.8 vs 나머지 6.0%p). 구 흐름이 더하는 것은 **시장 자체의 상승**(같은 기간 지수 +2.4 vs +0.1%)이라,
  두 조건이 겹친 거래가 그 뒤 가격이 가장 많이 올랐다 (평균 +8.2%).
- 주의: 싼 거래에는 저층 · 수리 필요 등 모델이 모르는 이유가 섞인다 (다음 거래도 적정가보다 쌌던 비율 40~44%). 화면에는 "모델 기준 적정가보다 싼 거래" 사실과
  과거 통계만 보여주고, 매수 권유 표현은 쓰지 않는다 (설계서 12장 Guardrail).

## 8. 서비스 지표 ③ · ④ — 정책 타임라인 · 권역 수급

- **정책 타임라인:** 03의 정책 이벤트(196건)와 이벤트 스터디를 지역 화면에 시간순으로 보여준다. 정책 Feature는 3개월 예측을 개선하지 못했지만(2 · 3장 `+ 정책(03)`),
  사건 직후 가격 · 거래량 변화는 분명했다 (아래 표).
- **권역 매매 · 전세 수급:** 부동산원 주간 수급동향(100 = 수요 · 공급 균형)은 구별 값이 없어 5개 권역 값으로 보여준다 (6장 주간 표에 포함).

In [11]:
pd.read_parquet(PROCESSED_DIR / "policy_event_study_v2.parquet")[["사건", "DiD(%)", "95% CI(%)", "사전추세 검정 p", "추세 보정 DiD(%)", "처치/대조 거래 비율 변화(%)"]].round(3)

,사건,DiD(%),95% CI(%),사전추세 검정 p,추세 보정 DiD(%),처치/대조 거래 비율 변화(%)
0,A. 잠삼대청 토허 해제 (2025-02-13),3.044,2.0 ~ 4.1,0.623,3.877,40.345
1,B. 강남3구·용산 토허 지정 (2025-03-24),3.218,2.4 ~ 4.0,0.000,-0.431,-66.593
2,C. 21개 구 규제지역 해제 (2023-01-05),-2.323,-3.7 ~ -1.0,0.153,0.469,-7.010


## 9. 산출물 저장

| 파일 | 내용 | 쓰임 |
|---|---|---|
| `gu_trend_weekly.parquet` | 구 × 주: 매매 4 · 13주 변화, 전세 4주 변화, 4주 순위, 권역 수급, 공표일 | 지역 화면 흐름 순위 · 차트 (주 1회 갱신) |
| `trend_baseline_monthly.csv` | 매달 4주 흐름 상위 8개의 이후 3개월 적중률 · 상승률 | "과거 적중률" 표시 |
| `opportunity_evidence.csv` | 구 흐름 × 적정가 대비 가격별 다음 거래 초과상승 | 기회 신호 설명 근거 |
| `display_config.json` | 표시 규칙 (흐름 순위 · 기회 신호 기준 · 문구 원칙) | 서빙 |
| `metrics.json` | 기준선 vs 모델 비교, 노출 결정 | 기록 |

In [12]:
trend_out = trend[["자치구", "권역", "주", "공표일", "매매_4주", "매매_13주", "전세_4주", "매매_4주_%", "매매_13주_%", "전세_4주_%",
                   "4주순위", "최근많이오른구", "매매수급", "전세수급"]].sort_values(["주", "4주순위"]).reset_index(drop=True)
trend_out.to_parquet(MODEL_DIR / "gu_trend_weekly.parquet", index=False)
monthly.assign(p=monthly["p"].astype(str)).round(4).to_csv(MODEL_DIR / "trend_baseline_monthly.csv", index=False, encoding="utf-8-sig")
cheap.to_csv(MODEL_DIR / "opportunity_evidence.csv", index=False, encoding="utf-8-sig")

display_config = {
    "region_probability": {"show": False, "reason": "어떤 모델도 Test에서 최근 4주 흐름 기준선을 넘지 못함 (6-2 노출 조건 미충족)"},
    "trend_rank": {"source": "부동산원 주간 아파트 매매가격지수 (KOSIS 408/DT_304004_WEEK_002_C)", "measure": "log(지수_t / 지수_t-4)",
                   "display": "(exp - 1) × 100 %", "top_n": 8, "label": "최근 4주 많이 오른 구", "refresh": "매주 목요일 (기준일 월요일 + 3일)",
                   "history": "trend_baseline_monthly.csv 연도별 적중률을 함께 표시"},
    "opportunity_signal": {"rule": "중개거래 11 적정가 대비 r = log(실거래 / 적정가) < log(0.95) (기존 단지, 등급 대상)", "context": "그 구의 4주 흐름 순위를 함께 표시",
                           "evidence": "opportunity_evidence.csv", "wording": "모델 기준 적정가보다 싼 거래 — 매수 권유 표현 금지"},
    "policy_timeline": {"source": "03 정책 이벤트 · policy_event_study_v2.parquet"},
    "supply_demand": {"source": "부동산원 주간 아파트 수급동향 (KOSIS 408/DT_304004_WEEK_013_C)", "unit": "서울 5개 권역", "neutral": 100},
}
json.dump(display_config, open(MODEL_DIR / "display_config.json", "w", encoding="utf-8"), ensure_ascii=False, indent=1)

pick = lambda d, period, name: d.loc[d["기간"].eq(period) & d["방법"].eq(name)].iloc[0]
metrics = {
    "target": "공식 주간 지수 순수 미래 3개월 변화 (상대: 같은 달 상위 8개 구)",
    "relative_top8": {name: {period: {"월내_AUC": round(float(pick(s4, period, name)["월내_AUC"]), 3),
                                      "상위8_적중률": round(float(pick(s4, period, name)["상위8_적중률"]), 3)} for period in ["DEV", "Test"]}
                      for name in ["기준선: 공식지수_4주변화", "로지스틱 창 전체 · 공식지수", "LightGBM 창 전체 · 위 + 회귀 OOF Gap"]},
    "trend_baseline_by_period": hit.round(3).to_dict("index"),
    "v1_target_reproduction_auc": round(float(s1.loc[0, "ROC_AUC"]), 3),
    "opportunity_excess_return_pctp": cheap.set_index(["구 흐름", "가격"])["초과상승(%p)"].unstack(0).round(2).to_dict(),
    "decision": "확률 숫자 미노출 → 4주 흐름 순위 · 기회 신호 · 정책 타임라인 · 권역 수급",
}
json.dump(metrics, open(MODEL_DIR / "metrics.json", "w", encoding="utf-8"), ensure_ascii=False, indent=1, default=float)
print(sorted(p.name for p in MODEL_DIR.iterdir()))

['display_config.json', 'gu_trend_weekly.parquet', 'metrics.json', 'opportunity_evidence.csv', 'trend_baseline_monthly.csv']


## 10. 정리

| 항목 | 결정 |
|---|---|
| 정답 | V1 중위 평당가 → 부동산원 공식 주간 지수 순수 미래 3개월 변화 (06) |
| 모델 | 로지스틱 · LightGBM · XGBoost × Feature 묶음 × 학습 창 — **채택 없음** (Test에서 최근 4주 흐름 기준선 미달) |
| 회귀 → 분류 연결 (OOF Gap) | Ablation에서 개선 없음 → 미사용. 11 → 31 연결은 모델 Feature가 아니라 **기회 신호**(화면)로 |
| 웹 지역 화면 | 확률 숫자 대신 4주 흐름 순위(과거 적중률 함께) · 기회 신호 · 정책 타임라인 · 권역 수급 |

### 남은 한계와 처리 계획

| 한계 | 처리 |
|---|---|
| Test 6개월 · DEV 26개월로 평가 기간이 짧다 (구 × 월 1,500행) | 서비스 후 매주 쌓이는 흐름 순위 적중률을 계속 기록 (`trend_baseline_monthly`) |
| 국면 전환(2026)에서 흐름 순위 적중률 52% | 문구를 "최근 많이 오른 구"(사실)로 제한, 연도별 적중률 함께 표시 |
| 기회 신호의 싼 거래에 층 · 수리 상태 등 미관측 요인 | 다음 거래도 쌌던 비율(40~44%)을 근거와 함께 기록, v3.1 사용자 입력 호가로 등급 판정 시 같은 기준 |
| 수급은 권역 단위뿐 | 구별 원천이 생기면 교체 |